# Outlines
* Required Installation
* Mathematical Background
  * Additive Model 
  * Uniqueness
  * Reliability
  * Uniformity
  * Modelling Attack: Logistic Regression
* XOR Arbiter PUF
  * Additive Delay Model
  * PyPUF Implementation
  * Security Analysis

## Required Installation

Run the cell below once to install the required packages. (Drop `--break-system-packages` if you're running inside a virtual environment — it's only needed on systems with an "externally managed" Python installation, e.g. recent Debian/Ubuntu.)

In [ ]:
%pip install pypuf==3.2.1 --no-deps --break-system-packages
%pip install tensorflow --break-system-packages
%pip install "numpy<2.0" --break-system-packages


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [57]:
import numpy as np

from pypuf.simulation import XORArbiterPUF
from pypuf.io import random_inputs
from pypuf.metrics import reliability, uniqueness, similarity_data

np.set_printoptions(precision=4, suppress=True)

## 1. Mathematical Background
### 1.1. Additive Delay Model
The Additive Delay Model is the general framework used across the whole family of **delay-based PUFs**.

**Core idea:** the physical quantity that decides the output (delay difference, phase difference, etc.) is a **linear combination** of a challenge-derived feature vector and a chip-specific weight vector:

$$\Delta = \mathbf{w}^T \boldsymbol{\phi}(\mathbf{c}) + w_0, \qquad r = \text{sign}(-\Delta)$$

- $\mathbf{w}$: fixed but unknown, chip-specific weights (encodes physical delay/phase variations)
- $\boldsymbol{\phi}(\mathbf{c})$: an **architecture-specific** transform of the challenge
- $w_0$: bias term

**Why this abstraction matters:**
1. **ML attack analysis generalizes** — any PUF fitting $\Delta = \mathbf{w}^T\phi+w_0$ is vulnerable to the same attack class (logistic regression, SVM), regardless of architecture.
2. **Security comparison reduces to one question:** how non-linear can $\phi(\mathbf{c})$ (or $\Delta$'s dependence on it) be made? XOR-PUF and Feed-Forward PUF are attempts to break this linearity without abandoning the additive skeleton.

**In short:** the Additive Delay Model abstracts any delay-based PUF as $\Delta = \mathbf{w}^T\phi(\mathbf{c})+w_0,\ r=\text{sign}(-\Delta)$; APUF is the special case where $\phi$ is the simple parity vector, while other architectures either redefine $\phi$ or combine multiple additive components non-linearly (e.g. via XOR).

### 1.2. Reliability
**Definition:** Measures how stable a single PUF instance's responses are when the same challenge is queried multiple times under simulated noise.

**Theoretical formula**:

$$\text{Reliability} = \Pr_{\text{eval}}\left[\text{eval}(x) = \text{eval}(x)\right]$$

That is, the probability that the same challenge $x$, evaluated twice under the noisy evaluation process, yields the same result.

In [58]:
# Reliability test.
#
# IMPORTANT: `instance` must have been created with `noisiness > 0`
# (e.g. ArbiterPUF(..., noisiness=0.05)). A noiseless instance
# (noisiness=0, the default) will always score a perfect 1.0, which is
# not informative.
#
# Returns the overall reliability score in [0, 1]. Ideal: 1.0.
def test_reliability(instance, seed: int = 10, N: int = 1000, r: int = 17,
                      verbose: bool = True) -> float:
    rel_per_challenge = reliability(instance, seed=seed, N=N, r=r)
    overall = float(np.average(rel_per_challenge))

    if verbose:
        print(f"[Reliability] Overall score : {overall:.4f}  (ideal: 1.0000)")
        print(f"[Reliability] Min observed  : {rel_per_challenge.min():.4f}")
        print(f"[Reliability] Max observed  : {rel_per_challenge.max():.4f}")

    return overall

### 1.3. Uniqueness

**Definition:** Measures how distinguishable a set of PUF instances (representing different physical chips of the same design) are from one another, based on their responses to the same set of random challenges.

**First, pairwise similarity between two chips is defined:**

$$\text{sim}(f,g) = \Pr_x\left[f(x) = g(x)\right] \approx \frac{1}{N}\sum_{i=1}^{N} \mathbb{1}[f(c_i) = g(c_i)]$$

**Uniqueness formula**, averaged over all chip pairs:

$$\text{Uniqueness} = 1 - 2 \cdot \mathbb{E}_{f \neq g}\left[\left|\tfrac{1}{2} - \text{sim}(f,g)\right|\right]$$

**Intuition:**
- Ideally, two different chips behave like independent coin flips on a random challenge, i.e. $\text{sim}(f,g) = 0.5$ → $|\tfrac{1}{2}-0.5|=0$ → Uniqueness $=1$.
- If two chips always agree ($\text{sim}=1$) or always disagree ($\text{sim}=0$), in both cases $|\tfrac{1}{2}-\text{sim}| = 0.5$ → Uniqueness $= 1 - 2(0.5) = 0$ (worst case: chips are indistinguishable).

For $l$ chips, the average is taken over all $\binom{l}{2}$ pairs.

In [59]:
# Uniqueness test
#
# `instances` must contain at least 2 PUF instances (e.g. created with
# different seeds, each representing a different "chip").
#
# Returns the overall uniqueness score in [0, 1]. Ideal: 1.0.
def test_uniqueness(instances: list, seed: int = 20, N: int = 2000,
                     verbose: bool = True) -> float:
    if len(instances) < 2:
        raise ValueError("Uniqueness test requires at least 2 PUF instances.")

    score = float(uniqueness(instances, seed=seed, N=N)[0])

    if verbose:
        print(f"[Uniqueness]  Score ({len(instances)} chips): {score:.4f}  "
              f"(ideal: 1.0000)")

    return score

### 1.4. Uniformity

**Definition:** Measures the balance of 0s and 1s in a single PUF instance's responses over a large random challenge set. Bias is measured by how far this value deviates from $0.5$.

$$\text{Uniformity} = \frac{1}{N}\sum_{i=1}^{N} r_i \quad, \quad r_i \in \{0,1\}$$



In [60]:
# Uniformity test.
#
# pypuf  has no function to calculate for this, so it is 
# computed directly: the fraction of responses equal
# to 1 (pypuf encodes responses as -1/+1).
#
# Returns the fraction of responses equal to 1. Ideal: 0.5.

def test_uniformity(instance, n: int, N: int = 5000, seed: int = 30,
                     verbose: bool = True) -> float:
    challenges = random_inputs(n=n, N=N, seed=seed)
    responses = instance.eval(challenges)  # values are -1 / +1

    responses_binary = (responses == 1).astype(int)
    score = float(responses_binary.mean())

    if verbose:
        deviation = abs(score - 0.5)
        print(f"[Uniformity]  Score: {score:.4f}  (ideal: 0.5000)  "
              f"-> {score*100:.2f}% ones / {(1-score)*100:.2f}% zeros, "
              f"deviation: {deviation:.4f}")

    return score

### 1.5. Modeling Attack (Logistic Regression Attack)

**Source of the weakness — additive delay model:**

$$\Delta = \mathbf{w}^T \boldsymbol{\phi}(\mathbf{c}) + w_0$$

Response:
$$r = \text{sign}(-\Delta) = \begin{cases}1 & \Delta<0\\0 & \Delta\geq0\end{cases}$$

**Attacker's goal:** without knowing the true $\mathbf{w}$, estimate $\hat{\mathbf{w}}$ from a set of $(\mathbf{c}_i, r_i)$ pairs.

**The logistic regression model** directly models the sign of $\Delta$ via a sigmoid function:

$$P(r=1 \mid \boldsymbol{\phi}) = \sigma(\hat{\mathbf{w}}^T \boldsymbol{\phi} + \hat{w}_0) = \frac{1}{1+e^{-(\hat{\mathbf{w}}^T \boldsymbol{\phi}+\hat{w}_0)}}$$

**Training minimizes the cross-entropy (log-loss):**

$$\mathcal{L}(\hat{\mathbf{w}}) = -\frac{1}{N}\sum_{i=1}^{N}\Big[r_i \log P(\boldsymbol{\phi}_i) + (1-r_i)\log\big(1-P(\boldsymbol{\phi}_i)\big)\Big] + \lambda\|\hat{\mathbf{w}}\|_2^2$$

**Why it works so well:** since $\Delta$ is a **linear** function of $\boldsymbol{\phi}(\mathbf{c})$, logistic regression's decision boundary ($\hat{\mathbf{w}}^T\boldsymbol{\phi}+\hat{w}_0 = 0$) belongs to the **exact same hypothesis class** as the PUF's true decision boundary ($\mathbf{w}^T\boldsymbol{\phi}+w_0=0$). This is why gradient descent can converge to $\hat{\mathbf{w}} \approx c\cdot\mathbf{w}$ (for some positive scaling constant $c$) with a finite number of CRPs.

**Attack success metric** (same formula as the similarity function):
$$\text{Attack Accuracy} = \frac{1}{N_{\text{test}}}\sum_{i=1}^{N_{\text{test}}} \mathbb{1}\big[\hat{r}_i = r_i\big]$$


In [61]:
import matplotlib.pyplot as plt
import tensorflow as tf
from pypuf.simulation.base import LTFArray
from pypuf.io import random_inputs, ChallengeResponseSet
from pypuf.attack import LRAttack2021
import pypuf.attack

# Logistic Regression modeling attack.
#
# Collects N challenge-response pairs, trains a logistic regression
# classifier on the parity-transformed challenges, and evaluates both
# train/test accuracy and similarity on a completely unseen challenge set.
#
# Returns a dict with keys: 'train_accuracy', 'test_accuracy', 'fresh_similarity'
def run_ml_attack(instance, n: int, k: int,N: int = 50000, split_ratio: float = 0.8,
                   train_seed: int = 2, fresh_seed: int = 5,
                   fresh_N: int = 2000, verbose: bool = True) -> dict:
    train_challenges = random_inputs(n=n, N=N, seed=train_seed)
    train_responses = instance.eval(train_challenges)
    test_challenges = random_inputs(n=n, N = N-(int)(N*split_ratio) , seed=fresh_seed)
    test_responses = instance.eval(test_challenges) 

    crps = ChallengeResponseSet(train_challenges[:N], train_responses[:N])
    attack = LRAttack2021(crps, seed=100, k=k, bs=1000, lr=0.001, epochs=100, stop_validation_accuracy=1.0)
    model = attack.fit()

    predicted = model.eval(test_challenges)
    accuracy = np.mean(
    predicted.reshape(-1) == test_responses.reshape(-1)
    )
    print('Attack accuracy:', accuracy)

## 2. XOR Arbiter PUF (A-PUF)

**Definition:** XOR Arbiter PUF combines $k$ **independent** classical APUF, each fed with the **same challenge**, by **XOR-ing** their outputs together.

### 2.1. Additive Delay Mode

First, we evaluate Arbiter PUF **Additive Delay Model**;

Let $\Delta_{i-1}$ be the delay difference entering stage $i$. The new difference $\Delta_i$ leaving that stage depends on the challenge bit $c_i$:

- **$c_i=0$ (straight):** signals don't swap → $\Delta_i = \Delta_{i-1} + \delta_i^{0}$
- **$c_i=1$ (crossed):** signals swap, so the previous difference's **sign flips** (the leading signal is now on the other line) → $\Delta_i = -\Delta_{i-1} + \delta_i^{1}$

Both cases combine into a single formula:

$$\Delta_i = (1-2c_i)\,\Delta_{i-1} + \delta_i(c_i), \qquad \delta_i(c_i) = (1-2c_i)\alpha_i + \beta_i$$

where:
- $\alpha_i$: the "straight-through" delay-difference component of that stage (chip-specific, due to manufacturing variation)
- $\beta_i$: a fixed delay component introduced by the crossover itself

tarting from $\Delta_0 = 0$ and unrolling for $i=1,\dots,n$:

$$\Delta_n = \sum_{i=1}^{n} \delta_i(c_i) \cdot \prod_{j=i+1}^{n}(1-2c_j)$$

Each stage's contribution ($\delta_i$) enters the sum multiplied by a coefficient that depends on **how many times all subsequent stages flip the sign**. This "product of subsequent stages" term is exactly the **parity vector**:

$$\phi_i = \prod_{j=i}^{n} (1-2c_j)$$

After algebraic simplification (grouping the $\alpha_i$ terms with $\phi_i$, and collapsing all $\beta_i$ terms into a single bias):

$$\boxed{\Delta = \mathbf{w}^T \boldsymbol{\phi}(\mathbf{c}) + w_0 = \sum_{i=1}^{n} \alpha_i \phi_i + w_0}$$

$$r = \begin{cases} 1 & \Delta < 0 \\ 0 & \Delta \geq 0 \end{cases}$$

where:
- $\mathbf{w}=(\alpha_1,\dots,\alpha_n)$: a chip-specific weight vector, randomly fixed at manufacturing time
- $\boldsymbol{\phi}(\mathbf{c})$: the architecture-specific parity vector computed from the challenge
- $w_0$: a constant bias term (sum of all $\beta_i$ contributions)

Following lines describe XOR-APUF  **Additive Delay Model**;

Each component ($l = 1, \dots, k$) follows the standard APUF additive model on its own:

$$\Delta^{(l)} = \mathbf{w}^{(l)T} \boldsymbol{\phi}(\mathbf{c}) + w_0^{(l)}$$

Each component's own response:
$$r^{(l)} = \text{sign}(-\Delta^{(l)})$$

The final response is their XOR:
$$r = r^{(1)} \oplus r^{(2)} \oplus \dots \oplus r^{(k)}$$

Since XOR becomes a **product** in the bipolar $\{-1,+1\}$ representation:

$$r' = \prod_{l=1}^{k} \text{sign}\left(-\Delta^{(l)}\right) = \prod_{l=1}^{k} \text{sign}\left(\mathbf{w}^{(l)T}\boldsymbol{\phi}(\mathbf{c}) + w_0^{(l)}\right)$$

A short comparision between Classical APUF and XOR APUF
| | Classical APUF | XOR APUF |
|---|---|---|
| $\Delta$ | A single linear sum: $\mathbf{w}^T\phi+w_0$ | **Product of the signs** of $k$ separate linear sums |
| Linearity | Fully linear in $\phi(\mathbf{c})$ | Linear within each component, but the combination is non-linear |
| ML resistance | Weak (broken by simple logistic regression) | Increases with $k$ (required CRP count grows exponentially) |
| Reliability | High | Decreases as $k$ grows (noise accumulates through the XOR) |
### 2.2. PyPUF Implementation
Create a 64-bit classical Arbiter PUF and generate a batch of challenge-response pairs (CRPs).

In [62]:
n = 64  # challenge length / number of MUX stages
k = 4
puf = XORArbiterPUF(n=n, k=4,seed=1)
print(f"[+] Simulated a {n}-bit classical XOR Arbiter PUF (seed=1)")

challenges = random_inputs(n=n, N=50000, seed=2)
responses = puf.eval(challenges)
print(f"[+] Generated {len(challenges)} challenge-response pairs")
print(f"    Example challenge (first 10 bits): {challenges[0][:10]}")
print(f"    Corresponding response           : {responses[0]}")

[+] Simulated a 64-bit classical XOR Arbiter PUF (seed=1)
[+] Generated 50000 challenge-response pairs
    Example challenge (first 10 bits): [-1 -1  1  1  1 -1  1  1  1  1]
    Corresponding response           : -1


## 2.3. Security Analysis
### 2.3.1. Reliability 

In [28]:
# Reliability requires a NOISY instance (same seed = same "chip",
# but with simulated environmental noise turned on)
noisy_puf = XORArbiterPUF(n=n, k=4, seed=1, noisiness=0.05)
test_reliability(noisy_puf, seed=10, N=1000, r=17)

[Reliability] Overall score : 0.9158  (ideal: 1.0000)
[Reliability] Min observed  : 0.5017
[Reliability] Max observed  : 1.0000


0.9158477508650519

### 2.3.2. Uniqueness

In [63]:
# Uniqueness requires MULTIPLE chip instances (different seeds)
chip_instances = [XORArbiterPUF(n=n, k=4, seed=s) for s in range(100, 110)]
test_uniqueness(chip_instances, seed=20, N=2000)

[Uniqueness]  Score (10 chips): 0.9818  (ideal: 1.0000)


0.9817555555555556

### 2.3.3. Uniqueness

In [64]:
# Uniformity is evaluated on a single chip instance
test_uniformity(puf, n=n, N=5000, seed=30)

[Uniformity]  Score: 0.4908  (ideal: 0.5000)  -> 49.08% ones / 50.92% zeros, deviation: 0.0092


0.4908

### 2.3.4. Modelling Attack: Logistic Regression

In [65]:
run_ml_attack(puf, n=n, k=k, N=50000)

Epoch 1/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4967 - loss: 0.6933 - val_accuracy: 0.5300 - val_loss: 0.6934
Epoch 2/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5019 - loss: 0.6930 - val_accuracy: 0.5180 - val_loss: 0.6935
Epoch 3/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5049 - loss: 0.6927 - val_accuracy: 0.5040 - val_loss: 0.6937
Epoch 4/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5125 - loss: 0.6924 - val_accuracy: 0.4820 - val_loss: 0.6941
Epoch 5/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5145 - loss: 0.6920 - val_accuracy: 0.4980 - val_loss: 0.6947
Epoch 6/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5148 - loss: 0.6916 - val_accuracy: 0.4880 - val_loss: 0.6954
Epoch 7/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5163 - loss: 0.6912 - val_accuracy: 0.4900 - val_loss: 0.6962
Epoch 8/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5167 - loss: 0.6908 - val_accuracy: 0.4900 - v

**WARNING:** If you encounter an error like; 


"Traceback (most recent call last):
  File "/home/seuz/QPUF Tutorial/deneme.py", line 27, in <module>
    model = attack.fit()
  File "/home/seuz/.local/lib/python3.10/site-packages/pypuf/attack/lr2021.py", line 145, in fit
    self._history = model.fit(
  File "/home/seuz/.local/lib/python3.10/site-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "/home/seuz/.local/lib/python3.10/site-packages/pypuf/attack/lr2021.py", line 89, in accuracy
    return tf.keras.metrics.binary_accuracy(.5 - .5 * y_true, .5 - .5 * y_pred)
TypeError: Expected int8, but got 0.5 of type 'float'."

run following cell and train again.

In [16]:
def keras_to_pypuf_fixed(self, keras_model):
    n, k = self.crps.challenges.shape[1], self.k
    weights = np.zeros(shape=(k, n))
    bias = np.zeros(shape=(k,))
    for l in range(k):
        layer_weights = keras_model.layers[l + 1].get_weights()
        weights[l] = layer_weights[0][:, 0]
        bias[l] = layer_weights[1][0]
    return LTFArray(weight_array=weights, bias=bias, transform=XORArbiterPUF.transform_atf)
pypuf.attack.LRAttack2021.keras_to_pypuf = keras_to_pypuf_fixed

@staticmethod
def loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    return tf.keras.losses.binary_crossentropy(.5 - .5 * y_true, .5 - .5 * y_pred)

@staticmethod
def accuracy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    return tf.keras.metrics.binary_accuracy(.5 - .5 * y_true, .5 - .5 * y_pred)

pypuf.attack.LRAttack2021.loss = loss
pypuf.attack.LRAttack2021.accuracy = accuracy